# Block 5 · Pattern discovery and text mining
*Which products go together, what is a complaint about, and how do we check that the answer can be trusted?*

> Data Mining · SGH Warsaw School of Economics (223121-0090)
>
> Version 2, updated 29 Oct 2026 for the class on 3 Nov. If Experiment 1 below is not called "Confidence against everyone", you have an older copy: update the course folder (see the setup guide).

Block 4 had no target column, and neither has tonight's first part: the rules come from the table itself, and nothing is predicted. In the second part a target returns: the topic of a complaint. Two parts:
1. Products held together: which products do customers hold together more often than chance gives? (association rules)
2. Texts: how does a complaint become a row of numbers, and which topic is it about? (TF-IDF and a classifier)

**The methods you practise tonight**
1. **Association rules** - a customer's **basket** is the set of products the customer holds: one row of a yes/no table. An **itemset** is a set of products; its **support** is the share of all customers who hold every product in it. For one product the support has its own name, **penetration**: the share of customers who hold it, also called its **base rate**. The **minimum support** is the bar an itemset must reach to be big enough to matter (ours: 3% of the customers); an itemset held by at least the minimum support is **frequent**, one below it **infrequent**. Itemsets are counted **level** by level: one product, pairs, sets of three, ... A bigger itemset that contains another is a **superset** of it, and it is held by at most as many customers (**downward closure**): so a superset of an infrequent itemset is never counted. Counting level by level and skipping such supersets is the **Apriori** algorithm. A **rule** A ⇒ B reads "customers who hold A often hold B too"; A is the left side (**antecedent**), B the right side (**consequent**). The rule's support is the support of A and B together; its **confidence** = customers with A and B / customers with A; its **lift** = confidence / B's penetration. Lift 1 means no link (A and B are **independent**); lift k means A's holders are k times as likely to hold B as everyone. A high confidence into a product almost everyone holds has a lift near 1: the **confidence trap**. **Shuffled baskets** (each product column shuffled on its own) show how high lift goes by chance.
2. **Text to numbers** - in books a text is a **document** and a word is a **token**. Split each text into words (lowercase; scikit-learn drops one-letter words such as "i" and "a"); the **vocabulary** is all the different words of all the texts. Very common words such as "the" are **stop words**; dropping a list of them is a separate option, not used tonight. The **bag of words** is the count table: one row per text, one column per word of the vocabulary, each cell = how often the word occurs in the text, its **tf** (term frequency). **df** (document frequency) = in how many texts a word occurs. **idf** (inverse document frequency), scikit-learn's formula: ln((1 + N) / (1 + df)) + 1, with N = the number of texts: the more texts contain a word, the lower its idf; a word found in every text gets 1 and stays in the table. **TF-IDF** = tf × idf; then each row is divided by its length (**normalising** it to length 1), so long and short texts are on one scale. A table that is mostly zeros is stored by its non-zero cells only: a **sparse** table. scikit-learn's **vectoriser** `TfidfVectorizer` does all of this in one object: it turns each text into a row of numbers.
3. **A text classifier** - the topics are its **classes** (five, instead of yes and no). Split the texts into training and test texts, as in Block 2; fit the vectoriser on the training texts only; a **pipeline** chains the vectoriser and the model in one object. Logistic regression with five classes gives every text five probabilities that add up to 1; the label is the topic with the largest probability, and that probability is the text's **top probability**. The **confusion matrix** counts the test texts by actual topic (rows) and predicted topic (columns). A topic's **recall** = the share of its texts that get its label; its **precision** = the share of the texts with its label that are about it. **Class balance**: how equally common the classes are; with a rare class, accuracy hides how badly that class is handled. A **confidence bar**: a text whose top probability reaches it is **labelled automatically**, the others are **sent to a person**; the bar is chosen on **out-of-fold** probabilities of the training texts (each text scored by a model that did not learn from it, as in Block 3) and checked once on the test texts. **Copies across the split**: the same text in training and in test makes the test look better than it is. **Drift**: the texts change after the model was trained; the model keeps labelling with no warning (a **silent failure**), so watch the share of texts below the bar.


**Our example data.** Two made-up data sets from the course's data loader, `finance_data.py` (nothing is downloaded):
* `load_baskets()`: 4,000 customers of the course's bank, one row per customer and one True/False column per product. The ten products: checking (an everyday account), debit card (pays from the account), savings (a savings account), credit card (pays on credit), personal loan (a loan for any purpose), home insurance (insurance of the customer's home), term deposit (savings locked in for a fixed time), mortgage (a loan to buy a home), car loan (a loan to buy a car) and brokerage (an account for buying shares). In the code the names have underscores: `debit_card`, `home_insurance`, ...
* `load_complaints()`: 1,200 complaints, 240 in each of five topics: app & service (the mobile app, transfers that fail, the hotline), card fraud (someone used, copied or stole a card), collections (calls and letters about a debt to repay), fees (charges on the account the customer did not expect) and mortgage (the home loan: its installments, rate and agreement). In the code: `app_service`, `card_fraud`, `collections`, `fees`, `mortgage`. The texts are lowercase, without punctuation, 15 to 67 words long; pln = Polish złoty. A separate set, with no link to the 4,000 customers.

Two small made-up worlds come first, small enough to compute everything by hand: **ten customers numbered 1-10** with four products (checking, savings, mortgage, home insurance) and **six short complaints S1-S6**. The data only illustrate the methods; the same steps work on any table of yes/no columns and any collection of texts.


**How we use this notebook.** The lecture stops five times for an **experiment**: the slide names it ("Experiment 1" ... "Experiment 5"); run that section. Each experiment has four parts:
* a **card** - the question, the one value you change, and what to look for (the same text as on the slide);
* a **plain cell** - change the value marked `# change me`, then re-run the cell (Shift+Enter);
* a **slider cell** - the same picture with a slider or a dropdown; it redraws when you let go of it. No slider? Use the plain cell;
* **The code behind it (for your project)** - the few mlxtend or scikit-learn lines behind the picture, explained, and a cell that runs them; read it at home.

**Tonight's experiments:** [1 - Confidence against everyone](#Experiment-1---Confidence-against-everyone) · [2 - Shuffle the baskets](#Experiment-2---Shuffle-the-baskets) · [3 - Spread a word](#Experiment-3---Spread-a-word) · [4 - Set the confidence bar](#Experiment-4---Set-the-confidence-bar) · [5 - A topic the model never saw](#Experiment-5---A-topic-the-model-never-saw). The table of contents (JupyterLab: the left sidebar; Colab: the list icon on the left) jumps to any section.

**Demos 1-7** show the slides' ideas computed (Demo 1 on the ten customers, the others on our data); replay them at home. The functions they call live in `block5_tools.py`, next to this notebook (they are the names with `lab.` in front); the slides are drawn by the same functions, so you see the same pictures. Most cells run in under a second; a few homework cells take some seconds. The homework is at the end: A by hand, B in code.

* `[*]` next to a cell means it is still running.
* A `NameError` (for example, name 'lab' is not defined) usually means the setup cell was not run, also after a restart: run the setup cell, then your cell again. To run everything above: JupyterLab *Run → Run All Above Selected Cell*; Colab *Runtime → Run before*.
* Decimals take a dot: 0.6. In Python, 0,6 means something else (a pair of numbers). In an answer cell, `length=2,907` stops the cell with `SyntaxError: positional argument follows keyword argument`, and `support=0,01` with a `SyntaxError` about "leading zeros": write 2.907 and 0.01.
* Text values take quotes: `"debit_card"`, not `debit_card`; without quotes Python stops with `NameError: name 'debit_card' is not defined`.
* **Google Colab** (nothing to install): open this notebook from GitHub, run the next cell (it fetches the two files below), then run the setup cell; nothing else changes. The data are made by `finance_data.py`, so nothing else is downloaded. If the sliders do not appear, use the plain cells.

```
!wget -nv -O finance_data.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_1/finance_data.py
!wget -nv -O block5_tools.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_5/block5_tools.py
```

Run the setup cell first: it must print `Setup OK`, then `Sliders OK` (or a line saying that you will use the plain cells - that is fine too). A line starting with "Note:" about the scikit-learn or mlxtend version needs no action now; only a WARNING needs the fix it names.

In [ ]:
# Google Colab only: fetch the course files (on your laptop this cell does nothing).
# Run it once, then run the setup cell below.
import sys
if "google.colab" in sys.modules:
    !wget -nv -O finance_data.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_1/finance_data.py
    !wget -nv -O block5_tools.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_5/block5_tools.py
else:
    print("Not on Colab: nothing to fetch. Run the setup cell below.")

In [ ]:
import sys
sys.path.append("../lecture_1")   # the course data loader, finance_data.py
sys.path.append(".")              # block5_tools.py, next to this notebook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", None)   # long table cells in full
pd.set_option("display.width", 120)

# You do not need to read the next lines: they stop with a clear message
# if a file is missing.
try:
    import block5_tools as lab    # tonight's experiments: one function each
except ModuleNotFoundError as error:   # a file is missing: say which one, and stop
    if error.name not in ("block5_tools", "finance_data"):
        raise
    import warnings; warnings.filterwarnings("ignore", "To exit")
    raise SystemExit(f"{error.name}.py was not found. Colab: run the cell "
                     "above first (if you did, check that wget printed no "
                     "ERROR line). Laptop: update the course folder (git pull, "
                     "see the setup guide).") from None

# Our two data sets. These are copies: changing them in your own cells does
# not change what the lab keeps.
baskets = lab.baskets().copy()         # 4,000 customers x 10 products, True / False
complaints = lab.complaints().copy()   # 1,200 complaints: text, category (the topic)

# Prints "Setup OK: ...", a Note about a package version if there is one
# (no action needed; only a WARNING needs the fix it names), then "Sliders OK"
# or a line saying that you will use the plain cells.
lab.setup_check()

## 1 Ten customers, four products (slide 8)
Ten made-up customers numbered 1-10 and four products: checking (an everyday account), savings (a savings account), mortgage (a loan to buy a home) and home insurance (insurance of that home). One row per customer, one column per product: 1 = holds it, 0 = does not. One row is the customer's **basket**: the set of products the customer holds. The customers have numbers, not letters, because A and B name the two sides of a rule. The slides compute every number of the rules on these ten by hand first. The cell prints how many customers hold each product and shows the table. `lab.TOY_BASKETS` is a copy: changing it in your own cells changes nothing in the lab.

In [ ]:
toy = lab.TOY_BASKETS.copy()      # customers 1-10 x four products, True / False
print("customers who hold each product:", toy.sum().to_dict())
toy.astype(int)                   # 1 = holds the product, 0 = does not

## Demo 1 - Rules by hand, computed (replay at home: slides 10-18)
The six steps of the recipe on the ten customers, done by the computer. Steps 1-2: Apriori level by level with a minimum support of 3 customers (30%): what is counted, what is skipped (a superset of an infrequent itemset) and what is kept. Steps 3-5: the eight rules from the four kept pairs, in the order of the slides (by lift, then support), each with its support, its confidence, the penetration of B, the lift, and the customers with A but not B (the ones a rule points to). Step 6: each column shuffled on its own, and the lift of mortgage ⇒ home insurance again.

In [ ]:
hand = lab.rules_by_hand()       # prints steps 1-2 level by level; returns the eight rules
display(hand)                    # steps 3-5

shuffled = lab.toy_shuffled(0)   # step 6: each column shuffled on its own
holders = shuffled["mortgage"].sum()
both = (shuffled["mortgage"] & shuffled["home_insurance"]).sum()
confidence = both / holders
lift = confidence / shuffled["home_insurance"].mean()
print(f"Shuffled: {both} of the {holders} mortgage holders hold home insurance: "
      f"confidence {confidence:.0%}, lift {lift:.2f}")

## 2 Our data (slide 20)
Our 4,000 made-up customers and their ten products, in plain words: checking, debit card, savings, credit card, personal loan, home insurance, term deposit, mortgage, car loan and brokerage (each one is explained at the top of this notebook). `baskets` has one row per customer and one True/False column per product, as the ten customers. The mean of a True/False column is the share of True values: the product's penetration. The cell prints how many products a customer holds on average, then each product's holders and penetration.

In [ ]:
print(baskets.shape)                                    # 4,000 customers x 10 products
print("products per customer, on average:", round(baskets.sum(axis=1).mean(), 1))
holders = pd.DataFrame({"customers": baskets.sum(),      # how many customers hold each product
                        "penetration": baskets.mean()})  # their share of all 4,000
holders.sort_values("customers", ascending=False)

## Demo 2 - Move the minimum support (replay at home: slide 21 and Backup 4, Apriori on our data)
Apriori on our 4,000 customers at a minimum support you choose. For each level (the number of products in an itemset) the bars show all possible itemsets, those Apriori counts and those it keeps; the title and the table give the totals and the rules (one or two products on the left, one on the right). Try 0.01, 0.03 (ours: at least 120 customers), 0.05 and 0.10.

In [ ]:
min_support = 0.03     # change me: 0.01 to 0.10
lab.support_bar(min_support)

## Experiment 1 - Confidence against everyone
*About 10 minutes.* Our 4,000 customers. Pick a product B, the right side; each bar is a rule A ⇒ B, one for each of the other nine products A.

**Question:** for which B are all nine confidences high, and is any of those confidences well above B's penetration? For which B does one rule stand far above the line?

**Change:** `right_side`: one of the ten products.

**Look for:** left, each rule's confidence (bars) and B's penetration, the share of all customers holding B, also called its base rate (dashed line); right, each rule's lift = bar / line (dashed at 1).

In [ ]:
right_side = "debit_card"   # change me: one of the ten products (lab.E1_PRODUCTS lists them)
lab.confidence_against_everyone(right_side)

In [ ]:
lab.slider(lab.confidence_against_everyone, right_side=lab.E1_PRODUCTS)   # a dropdown of the ten products

### The code behind it (for your project)
The computer finds every frequent itemset and every rule at once, with the Python package mlxtend.
```python
from mlxtend.frequent_patterns import apriori, association_rules
sets = apriori(baskets, min_support=0.03, use_colnames=True)
rules = association_rules(sets, metric="confidence", min_threshold=0.0)
rules.sort_values("lift", ascending=False)
```
* `baskets`: one row per customer, one True/False column per product (columns of 1 and 0 work too).
* `apriori` = steps 1-2: every itemset held by at least 3% of the customers, with its support; `use_colnames=True` shows the product names instead of column numbers.
* `association_rules` = steps 3-4: every rule from those itemsets. `antecedents` = the left side A, `consequents` = the right side B (each a set of products, Python's `frozenset`); `support` = the rule's support, A and B together; `consequent support` = B's penetration; `confidence`; `lift` = confidence / consequent support.
* `metric="confidence", min_threshold=0.0` keeps every rule; `min_threshold=0.5` would keep the rules with a confidence of at least 0.5.
* Sort with `rules.sort_values("lift", ascending=False)`, never by confidence alone (the confidence trap).
* `help(association_rules)` lists every argument and every metric. If your mlxtend asks for `num_itemsets`, add `num_itemsets=len(baskets)`.

The cell runs it, counts the itemsets and the rules (all of them, and those with one or two products on the left and one on the right, the ones the slides count), and shows the ten rules with one product on each side that have the highest lift (the sides written as text). A rule A ⇒ B and its reverse B ⇒ A always have the same lift and the same support, so the cell sorts by lift, then by support, then by A in alphabetical order: the table comes out in the same order every time.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
sets = apriori(baskets, min_support=0.03, use_colnames=True)
rules = association_rules(sets, metric="confidence", min_threshold=0.0)
small = rules[(rules["antecedents"].apply(len) <= 2) & (rules["consequents"].apply(len) == 1)]
print(f"{len(sets)} itemsets held by at least 3% of the customers; {len(rules):,} rules from them,")
print(f"{len(small)} with one or two products on the left and one on the right (the {len(small)} of the slides)")
one_each = rules[(rules["antecedents"].apply(len) == 1) & (rules["consequents"].apply(len) == 1)]
table = pd.DataFrame({"A": one_each["antecedents"].apply(lambda s: " + ".join(sorted(s))),
                      "B": one_each["consequents"].apply(lambda s: " + ".join(sorted(s))),
                      "support": one_each["support"], "confidence": one_each["confidence"],
                      "consequent support": one_each["consequent support"],
                      "lift": one_each["lift"]}).round(3)
# A => B and B => A have the same lift and support: then A in alphabetical order
table.sort_values(["lift", "support", "A"], ascending=[False, False, True]).head(10).reset_index(drop=True)

## Demo 3 - The rule to act on (replay at home: slide 24)
Step 5 on our data: four of our rules sorted by confidence, each with the customers who hold A and B, B's penetration, the lift and the customers with A but not B. Sorted by confidence, the trap comes first. Then the customers the chosen rule points to: mortgage holders without home insurance.

In [ ]:
display(lab.rule_table())    # four of our rules, sorted by confidence
targets = lab.customers_to_approach("mortgage", "home_insurance")   # a mortgage, no home insurance
print(len(targets), "customers to approach; the first five:")
targets.head()

## Experiment 2 - Shuffle the baskets
*About 8 minutes.* Our 4,000 customers with each product column shuffled on its own: every product keeps its penetration, and real links between products are gone. Then the rules that pass: support at least 3%.

**Question:** do rules still pass in shuffled baskets, and how high does their lift go?

**Change:** `shuffle`: "none" (our baskets) or 1 to 20 (a different shuffle each).

**Look for:** left, every rule that passes as a dot: support across, lift up (grey: our baskets); right, the best lift of every table you tried.

*Done early?* run the plain cell with `min_support = 0.01`: how high does the best shuffled lift go now?

In [ ]:
shuffle = "none"     # change me: "none" or 1 to 20
min_support = 0.03   # done early? 0.01
lab.shuffle_baskets(shuffle, min_support=min_support)   # lab.shuffle_baskets_reset() clears the tables you tried

In [ ]:
lab.slider(lab.shuffle_baskets, shuffle=lab.E2_SHUFFLES)   # a dropdown: "none", then shuffles 1 to 20

### The code behind it (for your project)
```python
rng = np.random.default_rng(1)
shuffled = pd.DataFrame({p: rng.permutation(baskets[p].to_numpy()) for p in baskets.columns})
```
* `np.random.default_rng(1)`: a random number generator; its number (the seed) fixes which shuffle, so the same shuffle comes back next time. Shuffle 1 of the experiment is seed 1.
* `rng.permutation(...)`: the column's values in a new random order. Each column keeps its values, so its penetration; the links between columns are gone, because each column is shuffled on its own.
* Then the same `apriori` and `association_rules` as in Experiment 1: whatever lift the shuffled table gives, chance gave it.

The cell runs one shuffle, checks that every penetration is unchanged, and counts the rules with one or two products on the left and one on the right, ours and shuffled.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
rng = np.random.default_rng(1)
shuffled = pd.DataFrame({p: rng.permutation(baskets[p].to_numpy()) for p in baskets.columns})
print("every penetration unchanged:", bool((shuffled.mean() == baskets.mean()).all()))
for name, table in [("ours", baskets), ("shuffle 1", shuffled)]:
    found = association_rules(apriori(table, min_support=0.03, use_colnames=True),
                              metric="confidence", min_threshold=0.0)
    found = found[(found["antecedents"].apply(len) <= 2) & (found["consequents"].apply(len) == 1)]
    print(f"{name:9s}: {len(found)} rules pass; best lift {found['lift'].max():.2f}")

## 3 Our complaints (slide 30)
Part 2. `complaints` has one row per complaint: `text` and `category`, its topic (the five topics are explained at the top of this notebook). The cell counts the texts per topic and their words, and shows three of them, drawn at random (`random_state=1` fixes which three).

In [ ]:
print(complaints.shape)                         # 1,200 complaints, two columns
print(complaints["category"].value_counts().sort_index())   # texts per topic
words = complaints["text"].str.split().str.len()
print(f"words per text: {words.min()} to {words.max()}, {words.mean():.1f} on average")
complaints.sample(3, random_state=1)

## 4 Six short complaints (slides 32-38)
The four steps of TF-IDF by hand, on six made-up complaints of one sentence each. The cell shows the six sentences; the bag of words (step 2: one row per sentence, one column per word, the counts); each word's df (in how many sentences it occurs) and idf (step 3, scikit-learn's formula ln((1 + 6) / (1 + df)) + 1); then, for S2 and for S1, each word's tf, df, idf, tf × idf and its weight after step 4, the row divided by its length.

In [ ]:
for i, sentence in enumerate(lab.SENTENCES, start=1):
    print(f"S{i}: {sentence}")
display(lab.toy_bag_of_words())   # step 2: the bag of words, 6 rows x 27 columns
display(lab.toy_idf())            # step 3: df and idf of each word
display(lab.weights_table(2))     # steps 3-4 for S2 (the line above the table gives its length)
lab.weights_table(1)              # the same for S1: 'my' occurs twice, so its tf is 2

## Experiment 3 - Spread a word
*About 8 minutes.* The six short complaints. The word 'twice' (now only in S2) is added to the end of more and more of the other sentences; S2's TF-IDF row is computed again each time.

**Question:** as 'twice' appears in more sentences, what happens to its idf and to its weight in S2? What is its idf when every sentence has it?

**Change:** `in_sentences`: 1 to 6.

**Look for:** left, each word's idf against the number of sentences that contain it ('twice' in red); right, S2's weights ('twice' in red).

In [ ]:
in_sentences = 2   # change me: 1 to 6
lab.spread_word(in_sentences)

In [ ]:
lab.slider(lab.spread_word, in_sentences=lab.E3_COUNTS)   # 1 to 6 sentences

### The code behind it (for your project)
```python
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
counts = CountVectorizer().fit_transform(texts)
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)
tfidf.get_feature_names_out()
tfidf.idf_
```
* `texts`: a list (or a column) of texts, one per row.
* `CountVectorizer()` does steps 1-2: the bag of words, one column per word of the vocabulary, in alphabetical order.
* `TfidfVectorizer()` does steps 1-4. `fit` learns the vocabulary and the idf from the texts; `transform` applies them, to these texts or to new ones; `fit_transform` does both.
* `X` is sparse: only the non-zero cells are stored. `X.toarray()` makes an ordinary table - for small tables only.
* `get_feature_names_out()`: the words, in the order of the columns; `idf_`: each word's idf.
* The defaults: lowercase; one-letter words dropped; no stop words removed (`stop_words="english"` would remove a list of them); `smooth_idf=True`, so idf = ln((1 + N) / (1 + df)) + 1; tf = the raw count; `norm="l2"`, each row scaled to length 1.

The cell adds 'twice' to the end of the five other sentences and prints its idf and S2's row.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
texts = [s if i == 1 else s + " twice" for i, s in enumerate(lab.SENTENCES)]   # i = 1 is S2
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)
words = tfidf.get_feature_names_out()
print("idf of 'twice':", round(float(tfidf.idf_[list(words).index("twice")]), 3))
s2 = pd.Series(X.toarray()[1], index=words)    # row 1 = S2 (Python counts from 0)
s2[s2 > 0].sort_values(ascending=False).round(3)

## Demo 4 - Our complaints as numbers (replay at home: slides 42-43)
The four steps on our 1,200 complaints. First the idf of twelve words, fitted on all 1,200 texts to describe the data: the common words weigh least. Then one complaint as the classifier will see it: its row of the TF-IDF table, with the vectoriser fitted on the 840 training texts (70% of the complaints; Demo 5 makes this split). A high weight means rare in the collection, not important for the topic: which words point to which topic is what the classifier learns.

In [ ]:
display(lab.idf_ours().round(2))    # twelve words: in how many texts, and their idf
lab.text_row(lab.EXAMPLE_TEXT).round(2)   # one complaint: its weights that are not 0

## Demo 5 - The classifier, steps 1-4 (replay at home: slides 44-51)
The recipe of the text classifier on our complaints. Step 1: Block 2's split (70% training, 30% test, stratified by topic, `random_state=42`). Step 2: the vectoriser learns the words from the training texts only. Step 3: logistic regression with five topics; the five probabilities of one complaint. Step 4, the test: the accuracy, the confusion matrix (rows = the actual topic, columns = the predicted one), each topic's recall and precision, and why the errors happen: texts that mix in a second topic (an aside), with three of the wrong texts.

In [ ]:
lab.classifier_report().round(3)   # prints steps 1-4, draws the confusion matrix; the table: each topic

### The code behind it (for your project)
```python
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
train_texts, test_texts, train_topics, test_topics = train_test_split(
    texts, topics, test_size=0.3, stratify=topics, random_state=42)
model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
model.fit(train_texts, train_topics)
model.predict_proba(test_texts)
confusion_matrix(test_topics, model.predict(test_texts))
print(classification_report(test_topics, model.predict(test_texts)))
```
* `train_test_split(..., test_size=0.3, stratify=topics, random_state=42)`: 30% test texts, each topic in the same share in both parts, the same split every time (as in Block 2).
* `make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))`: the vectoriser and the model in one object, a pipeline. `fit` fits the vectoriser on the training texts only, turns them into TF-IDF columns and fits the model; `predict` on new texts uses the training vocabulary and idf. `max_iter=2000` gives the fit more steps to finish.
* `predict_proba`: five probabilities per text, one column per topic in alphabetical order (`model.classes_`); `predict`: the topic with the largest.
* `confusion_matrix(actual, predicted)`: rows = the actual topic, columns = the predicted one, both alphabetical; the diagonal holds the right labels.
* `classification_report`: each topic's precision and recall, and the f1-score, which combines the two. Its `support` column is the number of test texts of the topic - not a rule's support.

The cell runs it and prints the accuracy, the confusion matrix and the report.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
texts, topics = complaints["text"], complaints["category"]
train_texts, test_texts, train_topics, test_topics = train_test_split(
    texts, topics, test_size=0.3, stratify=topics, random_state=42)
model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
model.fit(train_texts, train_topics)
predicted = model.predict(test_texts)
print(f"accuracy: {(predicted == test_topics).mean():.3f}")
print(confusion_matrix(test_topics, predicted))
print(classification_report(test_topics, predicted, digits=3))

## Demo 6 - A rare class (replay at home: slide 52)
**Class balance**: how equally common the classes are. Ours are balanced, 240 texts each. Suppose one topic were rare: the demo keeps 1 in 10 of its texts in training and in test (`sample(frac=0.1, random_state=42)`), trains again, and compares the accuracy and that topic's recall with the balanced model. It also prints the labels the rare topic's test texts get, and the result with `class_weight="balanced"`, which gives each topic the same total weight in training.

In [ ]:
rare = "fees"    # change me: one of the five topics
lab.rare_topic(rare)

## Experiment 4 - Set the confidence bar
*About 9 minutes.* Our 840 training texts, each with out-of-fold probabilities (five folds, as in Block 3). A text whose top probability reaches the bar is labelled automatically; the others go to a person.

**Question:** what is the lowest bar at which at most 2 in 100 automatic labels are wrong, and what share of the texts then goes to a person?

**Change:** `bar`: 0.30 to 0.90, in steps of 0.05.

**Look for:** left, the top probability of right labels (blue) and wrong labels (red), the bar dashed; right, the share wrong against the share labelled automatically, one dot per bar you tried.

In [ ]:
bar = 0.40   # change me: 0.30 to 0.90, in steps of 0.05
lab.confidence_bar(bar)   # lab.confidence_bar_reset() clears the bars you tried

In [ ]:
lab.slider(lab.confidence_bar, bar=lab.E4_BARS)   # 0.30 to 0.90, in steps of 0.05

### The code behind it (for your project)
```python
from sklearn.model_selection import StratifiedKFold, cross_val_predict
cv = StratifiedKFold(5, shuffle=True, random_state=42)
proba = cross_val_predict(model, train_texts, train_topics, cv=cv, method="predict_proba")
top = proba.max(axis=1)
decided = top >= 0.6
```
* `StratifiedKFold(5, shuffle=True, random_state=42)`: five folds of the training texts, each topic in the same share in every fold (Block 3).
* `cross_val_predict(..., method="predict_proba")`: each training text gets its five probabilities from the model fitted on the other four folds, a model that did not learn from it: the **out-of-fold** probabilities. The whole pipeline is fitted again inside every fold, the vectoriser included.
* `proba.max(axis=1)`: each text's top probability (the largest of its five); `top >= 0.6`: True = labelled automatically, False = sent to a person.
* The bar is chosen here, on the training texts; the test texts check it once. A bar chosen on the test texts would make that check look better than it is.

The cell computes the out-of-fold probabilities and the bar 0.60 on them, then makes that check with `lab.test_check(0.60)`: once on the 360 test texts - all of them, and the 174 that are not word-for-word copies of training texts (**copies across the split**: the made-up texts repeat).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
train, test = lab.split()            # the split of Demo 5: 840 training, 360 test texts
train_texts, train_topics = train["text"], train["category"]
model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=42)
proba = cross_val_predict(model, train_texts, train_topics, cv=cv, method="predict_proba")
top = proba.max(axis=1)                              # each text's top probability
label = np.array(sorted(train_topics.unique()))[proba.argmax(axis=1)]   # its label
decided = top >= 0.6                                 # True: labelled automatically
wrong = label[decided] != train_topics.to_numpy()[decided]
print(f"bar 0.60 on the training texts: {decided.mean():.1%} labelled automatically, "
      f"{wrong.mean():.1%} of them wrong\n")
lab.test_check(0.60)   # once, on the test texts: all 360, and the 174 that are not copies

## Experiment 5 - A topic the model never saw
*About 8 minutes.* The model trained again without one topic: its training texts are left out. Then it labels the 360 test texts, among them the 72 of the left-out topic. The bar is 0.60.

**Question:** which labels do the left-out topic's texts get, and how many of them fall below the bar?

**Change:** `left_out`: one of the five topics.

**Look for:** left, the labels its 72 texts get; right, the top probability of the familiar texts (blue) and of the left-out topic's texts (red), the bar dashed.

In [ ]:
left_out = "card_fraud"   # change me: "app_service", "card_fraud", "collections", "fees" or "mortgage"
lab.new_topic(left_out)

In [ ]:
lab.slider(lab.new_topic, left_out=lab.E5_TOPICS)   # a dropdown of the five topics

### The code behind it (for your project)
```python
keep = train_topics != "mortgage"
model.fit(train_texts[keep], train_topics[keep])
top = model.predict_proba(test_texts).max(axis=1)
(top < 0.6).mean()
```
* `keep`: True for the training texts of the other four topics; the model learns only those four, as if mortgage complaints did not exist yet.
* `predict_proba(test_texts)` now has four columns: the model can only answer with a topic it was trained on; there is no "none of these".
* `(top < 0.6).mean()`: the share of texts below the bar, the ones sent to a person. Computed for the left-out topic's texts and for the familiar ones, it shows the difference; on new texts, month by month, it is the number to watch (Demo 7).

The cell runs it with mortgage left out.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
train, test = lab.split()            # the split of Demo 5
train_texts, train_topics = train["text"], train["category"]
test_texts, test_topics = test["text"], test["category"]
keep = train_topics != "mortgage"    # the training texts of the other four topics
model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
model.fit(train_texts[keep], train_topics[keep])
print("topics the model knows:", list(model.classes_))
top = model.predict_proba(test_texts).max(axis=1)
new = (test_topics == "mortgage").to_numpy()        # the 72 mortgage test texts
print("labels of the 72 mortgage texts:",
      pd.Series(model.predict(test_texts[new])).value_counts().to_dict())
print(f"below the bar 0.60: mortgage texts {(top[new] < 0.6).mean():.1%}, "
      f"familiar texts {(top[~new] < 0.6).mean():.1%}")

## Demo 7 - Watch the share below the bar (replay at home: slide 60)
Step 6, watch. The model of Experiment 5, trained without mortgage, labels a stream of new texts: the 288 familiar test texts plus mortgage complaints, a share of the stream you choose (0 to 0.20). **Drift**: the texts change after the model was trained. The model keeps labelling with no warning, a **silent failure** - but the share of texts below the bar rises above its usual level. Count it every month.

In [ ]:
share_new = 0.10   # change me: 0, 0.05, 0.10 or 0.20
lab.watch_stream(share_new)

## Homework
Two kinds of items: **A** = practice by hand (a calculator is fine, and Python works as one) and **B** = code. Each A item repeats one of tonight's slides. Under each A item there is an answer cell: replace each `...` with your answer and run the cell; `lab.check` then says "right" or "not yet", with a hint (it never stops the notebook). `lab.check("A1")` with no answers lists what an item takes. The solutions are released later. Also answer the open exam-practice question from the last slides: *A city library has 1,000 members. Last year 200 of them borrowed a cookbook, 250 borrowed a gardening book, and 120 borrowed both. (a) Compute the support, the confidence and the lift of the rule cookbook ⇒ gardening book. (b) Say in one sentence what the lift means. (c) How many members borrowed a cookbook but no gardening book, and why are they the ones to approach?* (3-5 sentences); a model answer will be in the solutions.

### A - by hand

**A1. Count and keep** (slides 10-12). A sports club has eight members and four activities (1 = the member does it; `lab.HW_CLUB` holds the table):

| member | gym | swimming | yoga | tennis |
|---|---|---|---|---|
| 1 | 1 | 1 | 0 | 0 |
| 2 | 1 | 1 | 1 | 0 |
| 3 | 1 | 0 | 1 | 0 |
| 4 | 1 | 1 | 1 | 0 |
| 5 | 0 | 1 | 0 | 1 |
| 6 | 1 | 0 | 0 | 1 |
| 7 | 1 | 1 | 1 | 0 |
| 8 | 0 | 0 | 1 | 0 |

(a) The penetration of each activity. (b) With a minimum support of 3 members, which activities are frequent, and which pairs must be counted? (c) Which itemsets of three are counted? (d) How many itemsets are counted instead of 15, and how many are kept?

In [ ]:
# A1 - replace each ... with your answer, then run the cell.
# A list: [..., ..., ...]; an itemset as text: "... + ..."
lab.check("A1",
          penetration=...,       # gym, swimming, yoga, tennis (in %), as a list
          frequent=...,          # the frequent activities, a list of names
          pairs_counted=...,     # the pairs that must be counted, a list of itemsets
          triples_counted=...,   # the itemsets of three that are counted
          counted=...,           # how many itemsets are counted (of 15)
          kept=...)              # how many of them are kept

**A2. Confidence and lift** (slides 13-15, 17). A bookshop has 2,000 loyalty-card customers; 400 of them bought a travel guide, 300 a phrasebook, and 180 both. (a) The support and the confidence of travel guide ⇒ phrasebook. (b) The phrasebook's penetration and the lift, read in one sentence. (c) The confidence and the lift of phrasebook ⇒ travel guide. (d) Acting on travel guide ⇒ phrasebook, how many customers would you approach?

In [ ]:
# A2 - shares in % (9 for 9%) or as decimals (0.09)
lab.check("A2",
          support=...,               # travel guide => phrasebook
          confidence=...,
          penetration=...,           # the phrasebook's penetration
          lift=...,
          reverse_confidence=...,    # phrasebook => travel guide
          reverse_lift=...,
          approach=...)              # how many customers you would approach (travel guide => phrasebook)

**A3. Choose a rule** (slides 16-17, 24). A streaming service has 10,000 subscribers. For each rule: the viewers of the left side, the viewers of both sides, and the share of all subscribers who watch the right side.

| rule | viewers of the left side | viewers of both | share who watch the right side |
|---|---|---|---|
| drama ⇒ news | 4,000 | 3,280 | 81% |
| cooking ⇒ travel | 1,250 | 600 | 15% |
| opera ⇒ ballet | 140 | 98 | 10% |
| sport ⇒ films | 3,600 | 1,980 | 60% |

(a) The support, the confidence and the lift of each rule. (b) With a minimum support of 3%, which rule would you act on, and why not the other three? (c) How many subscribers would you approach?

In [ ]:
# A3 - lists in the order of the table: drama, cooking, opera, sport
lab.check("A3",
          support=...,      # the four supports (in %)
          confidence=...,   # the four confidences (in %)
          lift=...,         # the four lifts
          act_on=...,       # the rule you would act on, as text: "... => ..."
          approach=...)     # how many subscribers you would approach

**A4. Bag of words** (slides 33-36). Four short reviews of train journeys (`lab.HW_REVIEWS`): R1 "the train was late again", R2 "the seat was broken", R3 "the train was clean", R4 "the staff were kind". (a) The vocabulary. (b) The count table. (c) In how many reviews is each word?

In [ ]:
# A4 - the vocabulary as a list of words; df as a dict {"the": ..., ...};
# counts: four rows (R1-R4) of numbers, one per word, the words in alphabetical order
lab.check("A4",
          vocabulary=...,
          df=...,
          counts=...)

**A5. TF-IDF** (slides 36-38). The four reviews of A4. (a) The idf of the, was, train and clean (scikit-learn's formula with N = 4: idf = ln((1 + 4) / (1 + df)) + 1). (b) R3's tf × idf, and the length of that row. (c) R3's row normalised to length 1. (d) Which word gets idf 1, and is it removed?

In [ ]:
# A5 - Python as a calculator: np.log(7 / 2) + 1 is ln(7/2) + 1; np.sqrt(9) is 3
lab.check("A5",
          idf=...,        # the, was, train, clean (a list)
          length=...,     # the length of R3's row of tf x idf
          row=...,        # R3's row scaled to length 1 (the, was, train, clean)
          idf_one=...,    # the word with idf 1, as text
          removed=...)    # is it removed? True or False

**A6. Training texts only** (slides 46-47). A vectoriser is fitted on R1 and R2 of A4 only. (a) Its vocabulary. (b) Which words of R3 and of R4 does it keep? (c) Why must the test texts not be used to fit it?

In [ ]:
# A6 - lists of words; (c) is for yourself (the solutions answer it)
lab.check("A6",
          vocabulary=...,   # fitted on R1 and R2
          r3_keeps=...,     # the words of R3 it keeps
          r4_keeps=...)     # the words of R4 it keeps

**A7. Per class** (slides 49-50, 52). A classifier labels hotel reviews by topic: room, breakfast or location. Its confusion matrix on 100 test reviews (rows = the actual topic, columns = the predicted one; `lab.HW_CONFUSION`):

| actual \ predicted | room | breakfast | location |
|---|---|---|---|
| room | 40 | 6 | 4 |
| breakfast | 5 | 30 | 5 |
| location | 3 | 2 | 5 |

(a) The accuracy. (b) The recall and the precision of each topic. (c) Which topic is handled worst, and would the accuracy tell you?

In [ ]:
# A7 - decimals (0.80) or percents (80); recall and precision as lists in the order room, breakfast, location
lab.check("A7",
          accuracy=...,
          recall=...,
          precision=...,
          worst=...)      # the topic handled worst, as text

**A8. The confidence bar** (slides 53-56). Ten texts, each with its top probability and whether its label was right (R) or wrong (W): 0.95 R, 0.91 R, 0.88 R, 0.83 R, 0.72 R, 0.66 W, 0.58 R, 0.52 W, 0.45 W, 0.41 R (`lab.HW_TOP`). For the bars 0.5, 0.6 and 0.7: how many texts are labelled automatically, how many of them are wrong, and how many go to a person? Of these three bars, which is the lowest with no wrong automatic label?

In [ ]:
# A8 - lists of three numbers, one per bar: 0.5, 0.6, 0.7
lab.check("A8",
          labelled=...,     # labelled automatically
          wrong=...,        # wrong among them
          to_person=...,    # sent to a person
          lowest_bar=...)   # the lowest of the three bars with no wrong automatic label

**A9. A new topic** (slides 57-61). The hotel of A7 opens a spa, and reviews about it start to arrive; the classifier knows only room, breakfast and location. What happens to the spa reviews, would anyone notice, and what would you watch?

In [ ]:
# A9 - write your answer between the quotes (3-5 sentences)
answer_A9 = """
"""
lab.check("A9")

### B - code
Every item works on our data; the text items work on the training texts unless they say otherwise. The next cell imports the tools and loads the split of Demo 5: `train` and `test` hold the 840 training and the 360 test complaints (columns `text` and `category`). Also at hand: `baskets` and `complaints` from the setup cell; `side(...)`, which writes a side of a rule, a `frozenset`, as text; `lab.oof_proba()`, the out-of-fold probabilities of the 840 training texts (one column per topic, as in Experiment 4); `lab.copy_in_other_fold()`, True for a training text that has a word-for-word copy in another fold; `lab.rules(table, min_support)`, the rules of any True/False table with one or two products on the left and one on the right, to check your own.

In [ ]:
# Homework B: the tools (each one is explained in a "code behind it" cell above)
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold


def side(products):
    """A side of a rule as text: frozenset({'mortgage', 'car_loan'}) -> 'car_loan + mortgage'."""
    return " + ".join(sorted(products))


train, test = lab.split()      # the split of Demo 5: 840 training, 360 test complaints
print(len(train), "training texts,", len(test), "test texts")

**B1. Rules with mlxtend.** Run `apriori(baskets, min_support=0.03, use_colnames=True)` and `association_rules` (Experiment 1's code behind it). How many itemsets are there, and how many rules with one or two products on the left and one on the right? Show the 10 rules with one product on each side that have the highest lift. Then compute brokerage ⇒ term deposit by hand from the columns: its support, confidence and lift (the mean of a True/False column is a share).

In [ ]:
# B1 - your code
sets = apriori(baskets, min_support=0.03, use_colnames=True)
rules = association_rules(sets, metric="confidence", min_threshold=0.0)
small = rules[(rules["antecedents"].apply(len) <= 2) & (rules["consequents"].apply(len) == 1)]
# print len(sets) and len(small); then the ten one-each-side rules with the highest lift
both = baskets["brokerage"] & baskets["term_deposit"]    # True: the customer holds both
# support = both.mean(); confidence = both.sum() / baskets["brokerage"].sum(); lift = ...

**B2. The minimum support.** Count the rules (one or two products on the left, one on the right) at minimum supports of 1%, 3%, 5% and 10%. Which rules does a 1% bar add (support below 3%)? Show the five with the highest lift, with their customers (support × 4,000). Then shuffle the baskets 20 times (seeds 1 to 20, as in Experiment 2) and find the best lift of each shuffle at 1% and at 3%. In one sentence: why do small groups give higher lifts by chance?

In [ ]:
# B2 - your code
def count_rules(table, min_support):
    """The rules of a True/False table: one or two products on the left, one on the right."""
    found = association_rules(apriori(table, min_support=min_support, use_colnames=True),
                              metric="confidence", min_threshold=0.0)
    return found[(found["antecedents"].apply(len) <= 2) & (found["consequents"].apply(len) == 1)]


for min_support in [0.01, 0.03, 0.05, 0.10]:
    pass   # print min_support and len(count_rules(baskets, min_support))
# a shuffle: rng = np.random.default_rng(seed)
# shuffled = pd.DataFrame({p: rng.permutation(baskets[p].to_numpy()) for p in baskets.columns})

**B3. TF-IDF by hand, in code.** For S1, with numpy: the counts (`CountVectorizer`), df, idf (scikit-learn's formula, N = 6), tf × idf, the length of the row, and the row scaled to length 1. Compare with row 0 of `TfidfVectorizer().fit_transform(lab.SENTENCES)`.

In [ ]:
# B3 - your code
cv = CountVectorizer()
counts = cv.fit_transform(lab.SENTENCES).toarray()   # 6 x 27: steps 1-2
words = cv.get_feature_names_out()
df = (counts > 0).sum(axis=0)                        # in how many sentences each word occurs
# idf = np.log((1 + 6) / (1 + df)) + 1; then s1 = counts[0] * idf; its length; s1 / length

**B4. Word pairs, chosen on training texts.** Put `TfidfVectorizer(ngram_range=(1, 2))` in the pipeline: each pair of neighbouring words becomes an extra column. How many columns does it make on the training texts? Compare it with single words by 5-fold cross-validation on the training texts (`cross_val_score` with `StratifiedKFold(5, shuffle=True, random_state=42)`): the mean and the standard deviation of the five accuracies. Is the gap evidence (Block 3)? Only then, once, the test accuracy of both. Optional: Naive Bayes, the classic text baseline, `make_pipeline(CountVectorizer(), MultinomialNB())` (`from sklearn.naive_bayes import MultinomialNB`), in the same comparison.

In [ ]:
# B4 - your code
cv5 = StratifiedKFold(5, shuffle=True, random_state=42)
for ngrams in [(1, 1), (1, 2)]:
    model = make_pipeline(TfidfVectorizer(ngram_range=ngrams), LogisticRegression(max_iter=2000))
    scores = cross_val_score(model, train["text"], train["category"], cv=cv5)
    # print ngrams, scores.mean(), scores.std()
    # the columns: len(model.fit(train["text"], train["category"])[0].vocabulary_)

**B5. Copies across the split.** (a) How many test texts also occur, word for word, among the training texts (`test["text"].isin(set(train["text"]))`)? Fit the pipeline on the training texts and compute the accuracy on those copies and on the other test texts. (b) How many distinct texts are there among the 1,200 (`complaints["text"].nunique()`)? What would you do before splitting? (c) Choose the bar again on the out-of-fold probabilities of the training texts with no copy in another fold (`lab.oof_proba()`, `~lab.copy_in_other_fold()`): which of 0.60, 0.65 and 0.70 is the lowest with at most 2 in 100 automatic labels wrong? Then check it once on the test texts that are not copies.

In [ ]:
# B5 - your code
copy = test["text"].isin(set(train["text"]))     # True: the test text is also a training text
model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
model.fit(train["text"], train["category"])
right = model.predict(test["text"]) == test["category"]   # True: the label is right
# print copy.sum(), right[copy].mean(), right[~copy].mean(); complaints["text"].nunique()
proba = lab.oof_proba()                          # 840 x 5, out-of-fold
alone = ~lab.copy_in_other_fold()                # True: no copy in another fold
top = proba.max(axis=1)                          # each training text's top probability
oof_right = proba.idxmax(axis=1) == train["category"]   # was its out-of-fold label right?

**B6. A rare class and class weights.** Make app & service rare in training: keep 17 of its 168 training texts (`sample(frac=0.1, random_state=42)`) and all texts of the other topics; the test texts stay unchanged. Refit and compute each topic's recall on the test texts (`recall_score(..., average=None)` from `sklearn.metrics`, or the diagonal of the confusion matrix divided by the row totals). Then refit with `LogisticRegression(max_iter=2000, class_weight="balanced")`. Repeat with fees as the rare topic.

In [ ]:
# B6 - your code
rare = "app_service"
part = train[train["category"] == rare].sample(frac=0.1, random_state=42)   # 17 texts
smaller = pd.concat([train[train["category"] != rare], part])
# fit make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000)) on smaller,
# then with class_weight="balanced"; each topic's recall and the accuracy on test

## Keep learning - Block 5 reading
* Tan, Steinbach, Karpatne, Kumar - *Introduction to Data Mining* (2nd ed.), chapter 5 "Association Analysis": 5.1-5.3 (support, confidence, Apriori, rules) and 5.7.1 (lift, there called the interest factor). Free from the authors.
* Jurafsky & Martin - *Speech and Language Processing* (3rd ed. draft): chapter 4 (logistic regression for text classification: 4.3 and 4.7) and 11.1 (bag of words and tf-idf; its idf gives a word found in every text 0, scikit-learn's gives 1). Free online.
* James, Witten, Hastie, Tibshirani, Taylor - *An Introduction to Statistical Learning with Applications in Python* (ISLP), 10.4 Document Classification: a bag of words for film reviews; the rest of chapter 10 is next week. Free at statlearning.com.
* scikit-learn User Guide - *Text feature extraction* (bag of words, tf-idf, sparsity); *Pipelines*; *Model evaluation*: confusion matrix and classification report.